In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as settings
import numpy as np
import h5py
import ast

from astropy.convolution import convolve, Box1DKernel, Gaussian1DKernel

from utility.plot_from_hdf5 import plot_from_hdf5
from utility.plot_spectrum import plot_spectrum
from pipeline.processing.remove_spikes import remove_spikes
from pipeline.processing.fit_continuum import _powerlaw
from pipeline.processing.fit_gaussians import _gaussian, _two_gaussian
from pipeline.processing.scale_civ_to_nv import _civ_template
from pipeline.processing.parameters import CIV_FIT_WINDOW, LYA_NV_FIT_WINDOW, NV_LYA_CONTINUUM_WINDOWS, CIV_CONTINUUM_WINDOWS, CIV_AIR, NV_AIR, LYA_AIR
from utility.my_colors import palette_dark


DETAILS_PATH = "data/result_details.csv"
RELEVANT_RESULTS_PATH = "data/relevant_results.csv"
SPECTRA_PATH = "data/spectra.h5"

settings = {
        'font.size':16,
        'xtick.direction':'in', 
        'xtick.minor.visible':True,
        'xtick.top':True,
        'ytick.direction':'in', 
        'ytick.minor.visible':True,
        'ytick.right':True,
        'xtick.major.size':6.0,
        'xtick.minor.size':4.0,
        'ytick.major.size':6.0,
        'ytick.minor.size':4.0,
        'legend.frameon':False,
    }

plt.rcParams.update(**settings)

In [2]:
results = pd.read_csv(RELEVANT_RESULTS_PATH,index_col=0,dtype={"targetid":"str"})
details = pd.read_csv(DETAILS_PATH,dtype={"targetid":"str"},index_col=1)

In [3]:
core_erq_idx = results[(results["rew_civ"]>100) & (results["z-w3"]>3.9) & (results["z"]>2) & (results["z"]<3.4) & (results["snr_w3"]>3)].index

In [10]:
def plot_cell(targetid,ax,yticks_right=0,plot_mask=True,text_x=0.4,text_y=0.8,fontsize=13):
    targetid = str(targetid)

    with h5py.File(SPECTRA_PATH, "r") as f:
        data = f[f"{targetid}"][()]
        
    lam = data[0,:]
    flux = data[1,:]
    ivar = data[2,:]

    flux_clean, _ = remove_spikes(lam, flux, ivar=ivar)

    #-------------------------------------------------------------------------------------

    cont_params = ast.literal_eval(details.loc[targetid,"continuum_fit_lya_nv"])["fit_params"]
    cont_flux = _powerlaw(lam/1290,**cont_params)

    #-------------------------------------------------------------------------------------

    civ_params = ast.literal_eval(details.loc[targetid,"line_fit_civ"])
    civ_integration_window = ast.literal_eval(details.loc[targetid,"measurements_civ"])["integration_window"]

    if civ_params["accept_two"]:
        civ_profile = _two_gaussian(lam,civ_params["fit_params"]["amp_c"],civ_params["fit_params"]["cen_c"],civ_params["fit_params"]["sigma_c"],civ_params["fit_params"]["amp_w"],civ_params["fit_params"]["cen_w"],civ_params["fit_params"]["sigma_w"])
    else:
        civ_profile = _gaussian(lam, civ_params["fit_params"]["amplitude"],civ_params["fit_params"]["center"],civ_params["fit_params"]["sigma"])

    civ_cont_params = ast.literal_eval(details.loc[targetid,"continuum_fit_civ"])
    civ_cont_profile = civ_cont_params["fit_params"]["amplitude"] * ((lam / 1700.0) ** civ_cont_params["fit_params"]["alpha"])

    civ_profile += civ_cont_profile

    civ_mask = (lam > civ_integration_window[0]) & (lam < civ_integration_window[1])
    civ_lam = lam[civ_mask]
    civ_profile = civ_profile[civ_mask]

    #-------------------------------------------------------------------------------------

    nv_params = ast.literal_eval(details.loc[targetid,"line_fit_nv"])
    nv_integration_window = ast.literal_eval(details.loc[targetid,"measurements_nv"])["integration_window"]

    nv_profile = _civ_template(lam, nv_params["fit_params"]["scale"], nv_params["fit_params"]["shift"], civ_params["fit_params"], civ_params["accept_two"])+cont_flux

    nv_mask = (lam > nv_integration_window[0]) & (lam < nv_integration_window[1])
    nv_lam = lam[nv_mask]
    nv_profile = nv_profile[nv_mask]

    #-------------------------------------------------------------------------------------

    lya_params = ast.literal_eval(details.loc[targetid,"line_fit_lya"])
    lya_integration_window = ast.literal_eval(details.loc[targetid,"measurements_lya"])["integration_window"]

    if lya_params["accept_two"]:
        lya_profile = _two_gaussian(lam,lya_params["fit_params"]["amp_c"],lya_params["fit_params"]["cen_c"],lya_params["fit_params"]["sigma_c"],lya_params["fit_params"]["amp_w"],lya_params["fit_params"]["cen_w"],lya_params["fit_params"]["sigma_w"])+cont_flux
    else:
        lya_profile = _gaussian(lam, lya_params["fit_params"]["amplitude"],lya_params["fit_params"]["center"],lya_params["fit_params"]["sigma"])+cont_flux

    lya_mask = (lam > lya_integration_window[0]) & (lam < lya_integration_window[1])
    lya_lam = lam[lya_mask]
    lya_profile = lya_profile[lya_mask]

    #-------------------------------------------------------------------------------------

    if plot_mask:
        plot_mask = (lam < 1680) & (lam > 1150)
        ax.set_xlim(1150,1620)
    else:
        plot_mask = [True]*len(lam)
        ax.set_xlim(min(lam),max(lam))

    ax.plot(lam[plot_mask],convolve(flux_clean[plot_mask],Box1DKernel(5)),color="#222222",linewidth=0.5)

    ax.plot(lam[civ_mask],civ_profile,linewidth=1.5,label="CIV fit",color=palette_dark[1])
    ax.plot(lam[nv_mask],nv_profile,linewidth=1.5,label="NV fit",color=palette_dark[0])
    ax.plot(lam[lya_mask],lya_profile,linewidth=1.5,label=r"Ly$\alpha$ fit",color=palette_dark[2])

    if yticks_right == 1:
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")

    ax.text(text_x,text_y,f"{results.loc[targetid,'desiname']}",transform=ax.transAxes,fontsize=fontsize)
    ax.text(text_x,text_y-0.1,f"z={results.loc[targetid,'z']:.2f}, z-W3={results.loc[targetid,'z-w3']:.2f}",transform=ax.transAxes,fontsize=fontsize-1)
    ax.text(text_x,text_y-0.19,f"REW(CIV)={results.loc[targetid,'rew_civ']:.2f}",transform=ax.transAxes,fontsize=fontsize-1)

In [6]:
pretty_idxs = [1135,1995,2114,2014,1210,1287,1289,1279,1274,1486,130,1431,2,921,560,111]

In [52]:
fig, axes = plt.subplots(7,2,figsize=(14,21),sharex=True,dpi=500)

for i, ax in enumerate(axes.flatten()):
    plot_cell(targetid=core_erq_idx[pretty_idxs[i]],ax=ax,yticks_right=i%2)
    if i == 11:
        ax.set_yticklabels(["","0","5","10",""])

# remove whitespace between plots 
fig.subplots_adjust(
    left=0.1, right=0.98, bottom=0.1, top=0.92,
    wspace=0, hspace=0                             
)

fig.supxlabel(r"            Wavelength [$\AA$]",y=0.065)
fig.supylabel(r"    $F_{\lambda}~[10^{-17}~ergs~s^{-1}~cm^{-2}~{\AA}^{-1}]$",x=0.04)

handles, labels = axes[0,0].get_legend_handles_labels()

leg = fig.legend(
    reversed(handles), reversed(labels),
    loc='lower center',
    bbox_to_anchor=(0.52, 0.027), 
    ncols=4,
    frameon=False,
    fontsize=21
)

for line in leg.get_lines():
    line.set_linewidth(4.0)

# plt.savefig("images/image.png")

plt.show()

/tmp/ipykernel_32834/122814602.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(["","0","5","10",""])


In [15]:
settings = {
        'font.size':12,
        'xtick.direction':'in', 
        'xtick.minor.visible':True,
        'xtick.top':True,
        'ytick.direction':'in', 
        'ytick.minor.visible':True,
        'ytick.right':True,
        'xtick.major.size':6.0,
        'xtick.minor.size':4.0,
        'ytick.major.size':6.0,
        'ytick.minor.size':4.0,
        'legend.frameon':False,
    }

plt.rcParams.update(**settings)

In [93]:
for i, idx in enumerate(core_erq_idx):
    fig, ax = plt.subplots(figsize=(7,3),dpi=150)

    plot_cell(idx,ax=ax,plot_mask=False,text_x=0.67,text_y=0.9,fontsize=11)

    # plt.xlabel(r"Wavelength [$\AA$]",fontsize=11)
    # plt.ylabel(r"$F_{\lambda}~[10^{-17}~ergs~s^{-1}~cm^{-2}~{\AA}^{-1}]$",fontsize=11)

    plt.tight_layout()

    # handles, labels = ax.get_legend_handles_labels()

    # leg = fig.legend(
    #     reversed(handles), reversed(labels),
    #     loc='lower center',
    #     bbox_to_anchor=(0.5, -0.2), 
    #     ncols=4,
    #     frameon=False,
    #     fontsize=13
    # )

    # ax.text(0.03,-0.24,"Legend:",transform=ax.transAxes,fontsize=13)
    # ax.text(0.03,-0.34,r"x-axis:          Wavelength [$\AA$]",transform=ax.transAxes,fontsize=12)
    # ax.text(0.03,-0.44,r"y-axis:          $F_{\lambda}~[10^{-17}~ergs~s^{-1}~cm^{-2}~{\AA}^{-1}]$",transform=ax.transAxes,fontsize=12)

    plt.savefig(f"images/erq_spectra_plots/{idx}.jpg",facecolor="white", dpi=150)
    # plt.savefig(f"images/erq_spectra_plots/0header.jpg",facecolor="white", dpi=150)
    # print(idx)
    # plt.show()
    plt.close()
    
    if i%100==0:
        print(i)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
